# ENMV Model comparison (WAIC, PSIS-LOO)
This notebook compares fitted host-adaptation models with phenotype dimension \(n=1,2,3\) using posterior samples from MCMC inference. Model predictive performance is evaluated using **WAIC**, **PSIS-LOO cross-validation**, and an approximate **BIC**.

## Required files

The following files must be located in the same directory as this notebook:

- `Data_RESISTE_ENMV.xlsx` – experimental ENMV cross-inoculation data  
- `posterior_n1.pkl` – posterior samples for \(n=1\)  
- `posterior_n2.pkl` – posterior samples for \(n=2\)  
- `posterior_n3.pkl` – posterior samples for \(n=3\)


In [1]:
import os
import pickle
import math
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import erfcx

warnings.filterwarnings("ignore", message="Workbook contains no default style")

# ============================================================
# 0) DATA LOADING (standalone): rebuild HOSTS, H, Y, Y_CL, N_CL
# ============================================================
DATA_RESISTE = "Data_RESISTE_ENMV.xlsx"
assert os.path.exists(DATA_RESISTE), f"Missing {DATA_RESISTE}. Put it next to this notebook."

# Hosts (must match the order used in inference)
HOSTS = ["CH", "LA", "SA", "SO", "ZI"]
H = len(HOSTS)
host_to_idx = {h: i for i, h in enumerate(HOSTS)}

# Excel columns -> host codes
host_map = {
    "chicorée": "CH",
    "laitue": "LA",
    "salsifis": "SA",
    "souci": "SO",
    "zinnia": "ZI",
}

raw = pd.read_excel(DATA_RESISTE, sheet_name="ENMV")

needed = ["sp_inoc", "soucheV", "plante_source", "infected"]
missing = [c for c in needed if c not in raw.columns]
assert not missing, f"Missing columns: {missing}. Columns found: {list(raw.columns)}"

raw["target"] = raw["sp_inoc"].map(host_map)
raw["source"] = raw["soucheV"].astype(str)
raw["lineage"] = raw["plante_source"].astype(str)
raw["y"] = raw["infected"].astype(int)

valid_sources = ["7098MP1"] + HOSTS
raw = raw[raw["source"].isin(valid_sources)].copy()
raw = raw[raw["target"].isin(HOSTS)].copy()

# Split clonal vs evolved
df_clonal = raw[raw["source"] == "7098MP1"].copy()
df_evol = raw[raw["source"] != "7098MP1"].copy()

# Clonal aggregated counts per target
clonal_counts = df_clonal.groupby("target")["y"].agg(["sum", "count"]).reindex(HOSTS)
assert clonal_counts["count"].min() > 0, "Some clonal targets have zero observations (check Excel filtering)."

Y_CL = clonal_counts["sum"].astype(int).values
N_CL = clonal_counts["count"].astype(int).values

# Evolved lineage-level: each (source, lineage, target) has n=12
lin = df_evol.groupby(["source", "lineage", "target"])["y"].agg(["sum", "count"]).reset_index()
lin.rename(columns={"sum": "succ", "count": "n"}, inplace=True)
assert lin["n"].min() == 12 and lin["n"].max() == 12, "Expected 12 plants per (source, lineage, target)."

# Enforce 8 lineages per source (consistent order)
lineages_by_source = {s: sorted(lin[lin["source"] == s]["lineage"].unique()) for s in HOSTS}
for s in HOSTS:
    assert len(lineages_by_source[s]) == 8, f"Expected 8 lineages for source {s}, got {len(lineages_by_source[s])}"

# Build array Y[src, tgt, lineage] = successes (0..12)
Y = np.zeros((H, H, 8), dtype=int)
for src in HOSTS:
    for si, lin_id in enumerate(lineages_by_source[src]):
        sub = lin[(lin["source"] == src) & (lin["lineage"] == lin_id)]
        for tgt in HOSTS:
            y = int(sub[sub["target"] == tgt]["succ"].iloc[0])
            Y[host_to_idx[src], host_to_idx[tgt], si] = y

print("Data loaded:")
print("  HOSTS:", HOSTS)
print("  Y shape:", Y.shape, "(H,H,8)")
print("  Y_CL:", Y_CL, "N_CL:", N_CL)

# ============================================================
# 1) USER SETTINGS
# ============================================================
PKL_FILES = {
    1: "posterior_n1.pkl",
    2: "posterior_n2.pkl",
    3: "posterior_n3.pkl",
}

KAPPA_DEFAULT = 10.0
MAX_DRAWS = 4000
RNG_SEED = 123

# Numerical safety
EPS_P = 1e-6

# ============================================================
# 2) Load posterior pickles
# ============================================================
def load_posterior(path):
    assert os.path.exists(path), f"Missing posterior file: {path}"
    with open(path, "rb") as f:
        post = pickle.load(f)

    chains = post.get("chains", None)
    if chains is None:
        chains = post.get("results", None)
    assert chains is not None, f"Couldn't find 'chains' or 'results' in {path}"

    kappa = float(post.get("kappa", KAPPA_DEFAULT))

    n_dim = post.get("n_dim", None)
    assert n_dim is not None, f"Missing 'n_dim' in {path}"
    n_dim = int(n_dim)

    return post, chains, kappa, n_dim

def stack_samples(chains):
    return np.vstack([c["samples"] for c in chains])

rng = np.random.default_rng(RNG_SEED)

def maybe_subsample(S):
    if MAX_DRAWS is None or S.shape[0] <= MAX_DRAWS:
        return S
    idx = rng.choice(S.shape[0], size=MAX_DRAWS, replace=False)
    return S[idx]

models = {}
print("\nLoaded:")
for n_dim_expected, path in PKL_FILES.items():
    post, chains, kappa, n_dim = load_posterior(path)
    print(f"  n={n_dim_expected}: {path}  n_dim={n_dim}  kappa={kappa}  chains={len(chains)}")
    assert n_dim == n_dim_expected, f"{path}: expected n_dim={n_dim_expected}, got {n_dim}"
    S = maybe_subsample(stack_samples(chains))
    models[n_dim] = {
        "post": post,
        "chains": chains,
        "kappa": kappa,
        "S": S,
    }

# ============================================================
# 3) Mechanistic mapping
# ============================================================
def G_from_d2(d2, R, n_dim):
    R2 = R * R

    mu = 1.0 - (d2 + n_dim) / R2
    sig2 = (4.0 * d2 + 2.0 * n_dim) / (R2 * R2)
    sig = math.sqrt(max(sig2, 1e-15))

    z = mu / sig
    Phi_z = stats.norm.cdf(z)
    phi_z = stats.norm.pdf(z)

    G = mu * Phi_z + sig * phi_z
    return max(G, 0.0)

def L_from_d2(d2, R, n_dim):
    d = math.sqrt(max(d2, 0.0))

    if d < 1e-8:
        return float("inf")

    y = d + n_dim / (2.0 * R) - R
    z = y / math.sqrt(2.0)

    pref = (1.0 / R) * ((R / d) ** ((n_dim - 1.0) / 2.0))
    expo = math.exp(-0.5 * (d - R) * (d - R))

    # Stable version of:
    # sqrt(2/pi) - y * exp(y^2/2) * erfc(y/sqrt(2))
    bracket = math.sqrt(2.0 / math.pi) - y * erfcx(z)

    L = pref * expo * bracket

    if not math.isfinite(L):
        return float("inf")

    return max(L, 0.0)

def Pi_from_d2(d2, R, n_dim):
    G = G_from_d2(d2=d2, R=R, n_dim=n_dim)
    L = L_from_d2(d2=d2, R=R, n_dim=n_dim)
    return min(G, L)

def p_from_d2(d2, log10_U, R, qk, n_dim, eps_p=EPS_P):
    U = 10.0 ** log10_U
    R2 = R * R

    r = 1.0 - d2 / R2
    Pi_hat = Pi_from_d2(d2=d2, R=R, n_dim=n_dim)
    nu = 2.0 * U * Pi_hat

    inside = r + math.sqrt(2.0 * nu + 1.0) + math.sqrt(2.0 * nu + r * r) - 1.0
    rate = qk * max(inside, 0.0)

    p = 1.0 - math.exp(-rate)
    return min(max(p, eps_p), 1.0 - eps_p)

def build_p_matrix(O, log10_U, Rk, n_dim, qk):
    # O:  (H, n_dim) optima
    # Rk: (H,) one geometric scale per target host
    # qk: (H,) one target-specific q_k per target host
    # Returns:
    #   p_cl[k] for CL -> k
    #   p_evol[j, k] for source j -> target k
    if np.isscalar(Rk):
        Rk = np.full(H, float(Rk))
    else:
        Rk = np.asarray(Rk, float).reshape(-1)
        assert Rk.size == H, f"Expected Rk of length {H}, got {Rk.size}"

    qk = np.asarray(qk, float).reshape(-1)
    assert qk.size == H, f"Expected qk of length {H}, got {qk.size}"

    p_cl = np.zeros(H, float)
    p_evol = np.zeros((H, H), float)

    # CL is at origin, so d2 = ||O_k||^2
    for k in range(H):
        d2 = float(np.dot(O[k], O[k]))
        p = p_from_d2(d2, log10_U, float(Rk[k]), float(qk[k]), n_dim)
        p_cl[k] = min(max(p, EPS_P), 1.0 - EPS_P)

    # evolved: distances between optima
    for j in range(H):
        for k in range(H):
            d = O[j] - O[k]
            d2 = float(np.dot(d, d))
            p = p_from_d2(d2, log10_U, float(Rk[k]), float(qk[k]), n_dim)
            p_evol[j, k] = min(max(p, EPS_P), 1.0 - EPS_P)

    return p_cl, p_evol

def theta_to_params(theta, n_dim):
    theta = np.asarray(theta, float)

    expected_len = 1 + H + H + H * n_dim
    assert theta.size == expected_len, (
        f"Wrong theta length for n_dim={n_dim}: got {theta.size}, "
        f"expected {expected_len} = 1 + H + H + H*n_dim"
    )

    # theta = (log10_U, logR_k[1..H], q_k[1..H], O_flat)
    pos = 0

    log10_U = float(theta[pos])
    pos += 1

    logRk = theta[pos:pos + H]
    pos += H
    Rk = np.exp(logRk)

    qk = theta[pos:pos + H]
    pos += H

    O = theta[pos:].reshape(H, n_dim)

    return log10_U, Rk, qk, O

# ============================================================
# 4) Pointwise log-likelihood
# ============================================================
def loglik_pointwise(theta, n_dim, kappa):
    log10_U, Rk, qk, O = theta_to_params(theta, n_dim)

    p_cl, p_evol = build_p_matrix(O, log10_U, Rk, n_dim, qk)

    ll_parts = []

    # Clonal block: H binomial observations
    ll_cl = stats.binom.logpmf(Y_CL, N_CL, np.clip(p_cl, EPS_P, 1.0 - EPS_P))
    ll_parts.append(ll_cl.astype(float))

    # Evolved block: H*H*8 beta-binomial observations
    ll_lin = []
    for j in range(H):
        for k in range(H):
            p = float(np.clip(p_evol[j, k], EPS_P, 1.0 - EPS_P))
            a = kappa * p
            b = kappa * (1.0 - p)
            ll_lin.append(stats.betabinom.logpmf(Y[j, k, :], 12, a, b))
    ll_parts.append(np.concatenate(ll_lin).astype(float))

    return np.concatenate(ll_parts)

def compute_loglik_matrix(S, n_dim, kappa):
    return np.vstack([loglik_pointwise(S[i], n_dim, kappa) for i in range(S.shape[0])])

for n_dim, m in models.items():
    m["L"] = compute_loglik_matrix(m["S"], n_dim, m["kappa"])
    print(f"\nloglik matrix n={n_dim}: {m['L'].shape}")

# ============================================================
# 5) Posterior summaries for comparison across n
# ============================================================
def summarize_vector(x, labels):
    x = np.asarray(x, float)
    if x.ndim == 1:
        x = x[:, None]

    rows = []
    for j in range(x.shape[1]):
        rows.append({
            "param": labels[j],
            "mean": float(np.mean(x[:, j])),
            "sd": float(np.std(x[:, j], ddof=1)),
            "q2.5": float(np.quantile(x[:, j], 0.025)),
            "q50": float(np.quantile(x[:, j], 0.50)),
            "q97.5": float(np.quantile(x[:, j], 0.975)),
        })
    return pd.DataFrame(rows)

posterior_summaries = {}

for n_dim, m in models.items():
    S = m["S"]

    log10U = S[:, 0]
    logRk = S[:, 1:1 + H]
    Rk = np.exp(logRk)
    qk = S[:, 1 + H:1 + 2 * H]
    O_flat = S[:, 1 + 2 * H:]
    O = O_flat.reshape(S.shape[0], H, n_dim)

    dfs = []

    dfs.append(summarize_vector(log10U, ["log10_U"]))
    dfs.append(summarize_vector(Rk, [f"R_{h}" for h in HOSTS]))
    dfs.append(summarize_vector(qk, [f"q_{h}" for h in HOSTS]))

    O_cols = []
    O_names = []
    for h_idx, h in enumerate(HOSTS):
        for d in range(n_dim):
            O_cols.append(O[:, h_idx, d])
            O_names.append(f"O_{h}_{d + 1}")
    O_mat = np.column_stack(O_cols)
    dfs.append(summarize_vector(O_mat, O_names))

    summary_df = pd.concat(dfs, ignore_index=True)
    summary_df.insert(0, "n_dim", n_dim)
    posterior_summaries[n_dim] = summary_df

summary_all = pd.concat([posterior_summaries[n] for n in sorted(posterior_summaries)], ignore_index=True)

print("\nPosterior summaries (first rows):")
print(summary_all.head(20))

# Compact comparisons
log10U_compare_rows = []
R_compare_rows = []
q_compare_rows = []

for n_dim, m in models.items():
    S = m["S"]

    log10U = S[:, 0]
    logRk = S[:, 1:1 + H]
    Rk = np.exp(logRk)
    qk = S[:, 1 + H:1 + 2 * H]

    log10U_compare_rows.append({
        "n_dim": n_dim,
        "param": "log10_U",
        "mean": float(np.mean(log10U)),
        "sd": float(np.std(log10U, ddof=1)),
        "q2.5": float(np.quantile(log10U, 0.025)),
        "q50": float(np.quantile(log10U, 0.50)),
        "q97.5": float(np.quantile(log10U, 0.975)),
    })

    for k, host in enumerate(HOSTS):
        R_compare_rows.append({
            "n_dim": n_dim,
            "host": host,
            "mean": float(np.mean(Rk[:, k])),
            "sd": float(np.std(Rk[:, k], ddof=1)),
            "q2.5": float(np.quantile(Rk[:, k], 0.025)),
            "q50": float(np.quantile(Rk[:, k], 0.50)),
            "q97.5": float(np.quantile(Rk[:, k], 0.975)),
        })
        q_compare_rows.append({
            "n_dim": n_dim,
            "host": host,
            "mean": float(np.mean(qk[:, k])),
            "sd": float(np.std(qk[:, k], ddof=1)),
            "q2.5": float(np.quantile(qk[:, k], 0.025)),
            "q50": float(np.quantile(qk[:, k], 0.50)),
            "q97.5": float(np.quantile(qk[:, k], 0.975)),
        })

log10U_compare = pd.DataFrame(log10U_compare_rows)
R_compare = pd.DataFrame(R_compare_rows)
q_compare = pd.DataFrame(q_compare_rows)

print("\nComparison for log10_U:")
print(log10U_compare)

print("\nComparison for R_k:")
print(R_compare)

print("\nComparison for q_k:")
print(q_compare)

# ============================================================
# 6) WAIC
# ============================================================
def logmeanexp(a, axis=0):
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    return (m + np.log(np.mean(np.exp(a - m), axis=axis, keepdims=True))).squeeze(axis)

def waic_from_loglik(L):
    # L: (n_draws, n_obs)
    lppd_i = logmeanexp(L, axis=0)
    p_waic_i = np.var(L, axis=0, ddof=1)
    elpd_i = lppd_i - p_waic_i
    elpd = float(np.sum(elpd_i))
    se = float(np.sqrt(L.shape[1] * np.var(elpd_i, ddof=1)))
    waic = float(-2.0 * elpd)
    return {"elpd": elpd, "se": se, "waic": waic, "elpd_i": elpd_i}

for n_dim, m in models.items():
    m["waic"] = waic_from_loglik(m["L"])

print("\nWAIC:")
for n_dim in sorted(models):
    w = models[n_dim]["waic"]
    print(f"  n={n_dim}: elpd={w['elpd']:.2f}  SE={w['se']:.2f}  WAIC={w['waic']:.2f}")

def delta_summary(a, b):
    d_i = b["elpd_i"] - a["elpd_i"]
    d = float(np.sum(d_i))
    se = float(np.sqrt(d_i.size * np.var(d_i, ddof=1)))
    return d, se

print("\nPairwise Δelpd_waic (B - A):")
pairs = [(1, 2), (1, 3), (2, 3)]
for a, b in pairs:
    d, se = delta_summary(models[a]["waic"], models[b]["waic"])
    print(f"  n={b} - n={a}: {d:.2f}  SE={se:.2f}")

# ============================================================
# 7) PSIS-LOO (standalone, no ArviZ dependency)
# ============================================================
def _logsumexp(x, axis=None):
    x = np.asarray(x, float)
    m = np.max(x, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return out.squeeze(axis)

def psis_smooth_weights(logw, tail_frac=0.2, min_tail=10):
    """
    Pareto-smoothed importance sampling (PSIS) for one set of log-weights.
    """
    lw = np.asarray(logw, float).copy()
    S = lw.size
    if S < 5:
        return lw, np.nan

    lw -= np.max(lw)
    w = np.exp(lw)

    M = int(np.ceil(tail_frac * S))
    M = max(M, min_tail)
    M = min(M, S - 1)

    idx = np.argsort(w)
    w_sorted = w[idx]
    tail_idx = idx[-M:]
    w_tail = w[tail_idx]

    u = w_sorted[-M]
    excess = w_tail - u
    excess = np.maximum(excess, 0.0)

    if np.all(excess <= 0) or np.max(excess) <= 0:
        return lw, 0.0

    try:
        k_hat, loc_hat, sig_hat = stats.genpareto.fit(excess, floc=0.0)
        if not np.isfinite(k_hat) or not np.isfinite(sig_hat) or sig_hat <= 0:
            return lw, np.nan
    except Exception:
        return lw, np.nan

    j = np.arange(1, M + 1)
    p = (j - 0.5) / M
    q_excess = stats.genpareto.ppf(p, c=k_hat, loc=0.0, scale=sig_hat)
    q_excess = np.maximum(q_excess, 0.0)
    w_tail_smooth = u + q_excess

    w_max = np.max(w)
    w_tail_smooth = np.minimum(w_tail_smooth, w_max)

    w_smooth = w.copy()
    tail_sorted_order = np.argsort(w_tail)
    w_smooth[tail_idx[tail_sorted_order]] = w_tail_smooth

    logw_smooth = np.log(np.maximum(w_smooth, 1e-300))
    return logw_smooth, float(k_hat)

def psis_loo_from_loglik(L, tail_frac=0.2):
    """
    Compute PSIS-LOO from a log-likelihood matrix L (draws x obs).
    """
    L = np.asarray(L, float)
    S, N = L.shape
    elpd_i = np.zeros(N, float)
    pareto_k = np.zeros(N, float)

    for i in range(N):
        ll_i = L[:, i]

        # Raw log-weights for LOO: w_s ∝ 1 / p(y_i | theta_s)
        logw_raw = -ll_i

        logw_smooth, k_hat = psis_smooth_weights(logw_raw, tail_frac=tail_frac)
        pareto_k[i] = k_hat

        logw_norm = logw_smooth - _logsumexp(logw_smooth, axis=0)

        # log(sum_s w_s * p(y_i | theta_s))
        elpd_i[i] = _logsumexp(logw_norm + ll_i, axis=0)

    elpd = float(np.sum(elpd_i))
    se = float(np.sqrt(N * np.var(elpd_i, ddof=1)))
    looic = float(-2.0 * elpd)
    return {"elpd": elpd, "se": se, "looic": looic, "elpd_i": elpd_i, "pareto_k": pareto_k}

for n_dim, m in models.items():
    m["loo"] = psis_loo_from_loglik(m["L"], tail_frac=0.2)

print("\nPSIS-LOO (standalone):")
for n_dim in sorted(models):
    loo = models[n_dim]["loo"]
    print(f"  n={n_dim}: elpd={loo['elpd']:.2f}  SE={loo['se']:.2f}  LOOIC={loo['looic']:.2f}")

print("\nPairwise Δelpd_loo (B - A):")
for a, b in pairs:
    d_i = models[b]["loo"]["elpd_i"] - models[a]["loo"]["elpd_i"]
    d = float(np.sum(d_i))
    se = float(np.sqrt(d_i.size * np.var(d_i, ddof=1)))
    print(f"  n={b} - n={a}: {d:.2f}  SE={se:.2f}")

print("\nPareto k summaries (diagnostics):")
for n_dim in sorted(models):
    k = models[n_dim]["loo"]["pareto_k"]
    print(
        f"  n={n_dim}: "
        f"max={float(np.nanmax(k)):.3f}  "
        f">0.7={int(np.sum(k > 0.7))}/{k.size}  "
        f">1.0={int(np.sum(k > 1.0))}/{k.size}"
    )


Data loaded:
  HOSTS: ['CH', 'LA', 'SA', 'SO', 'ZI']
  Y shape: (5, 5, 8) (H,H,8)
  Y_CL: [19 16 21  0  4] N_CL: [20 20 40 60 20]

Loaded:
  n=1: posterior_n1.pkl  n_dim=1  kappa=10.0  chains=10
  n=2: posterior_n2.pkl  n_dim=2  kappa=10.0  chains=10
  n=3: posterior_n3.pkl  n_dim=3  kappa=10.0  chains=10

loglik matrix n=1: (4000, 205)

loglik matrix n=2: (4000, 205)

loglik matrix n=3: (4000, 205)

Posterior summaries (first rows):
    n_dim    param      mean        sd      q2.5       q50      q97.5
0       1  log10_U -0.655383  0.296502 -1.305788 -0.588257  -0.190044
1       1     R_CH  3.309235  3.103457  0.175192  1.939188  10.897507
2       1     R_LA  5.814331  2.693159  0.918514  5.488905  11.510592
3       1     R_SA  7.287497  2.931204  2.277858  7.045836  11.782491
4       1     R_SO  0.059800  0.095078  0.010495  0.019367   0.353171
5       1     R_ZI  0.549776  0.386499  0.225520  0.380287   1.617844
6       1     q_CH  4.283142  3.386011  1.023583  1.768992   9.762144
7 